In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

import optuna

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("datasets/kingametric_credit_risk.csv")

In [4]:
df.shape

(8744, 45)

In [5]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [6]:
cat_cols = ["Payment_of_Min_Amount", "Credit_Mix", "Payment_Behaviour", "Borrower_Tier"]

df[cat_cols] = df[cat_cols].astype("category")

In [7]:
X = df.drop(columns=["Default_Flag"], axis=1)
y = df["Default_Flag"]

In [8]:
folds = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

In [10]:
#low_var_cols = [col for col in X.columns if X[col].unique() < 5]
#X = X.drop(columns=low_var_cols)

In [12]:
base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    enable_categorical=True, 
    tree_method="hist",
    learning_rate=0.05,
    random_state=42
)
base_model.fit(X, y)

importances = base_model.feature_importances_

feature_importance = (
    pd.Series(importances, index=X.columns).sort_values(ascending=False)
)

In [13]:
top_features = feature_importance.head(20).index

X_select = X[top_features]